# Q2 — The Midnight Episode: Catching the Arrhythmia
**Real Dataset:** `patient_ecg.npy` (5000 samples) · `template.npy` (200 samples)

**Result:** Arrhythmia onset detected at **m = 2400, t = 9.60 s**

---

## 0 · Imports & Global Style

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import spectrogram as sp_spectrogram, butter, filtfilt

np.random.seed(42)

# ── Signal parameters ─────────────────────────────────────────────────────────
FS  = 250    # sampling frequency (Hz)
N   = 5000   # total samples → 20 seconds
L   = 200    # one healthy beat → 0.8 s

# ── Colour palette ────────────────────────────────────────────────────────────
C_HEALTHY  = '#00e5ff'
C_ARRHY    = '#ff4081'
C_TEMPLATE = '#69ff47'
C_ONSET    = '#ffea00'
C_TEXT     = '#e0e0e0'

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#13132b',
    'axes.edgecolor':   '#3a3a5e', 'axes.labelcolor': C_TEXT,
    'xtick.color': C_TEXT,         'ytick.color':     C_TEXT,
    'text.color':  C_TEXT,         'grid.color':      '#2a2a3e',
    'grid.linewidth': 0.5,         'legend.facecolor': '#1e1e3a',
    'legend.edgecolor': '#3a3a5e', 'axes.titlesize': 10,
    'axes.labelsize': 9,           'legend.fontsize': 8,
})
print('Setup complete.')


---
## 1 · Load Real Dataset

In [ ]:
# Load the provided real ECG recording and template
ecg_signal = np.load('patient_ecg.npy')   # 5000-sample ECG at 250 Hz
template   = np.load('template.npy')       # 200-sample clean healthy-beat template
time_axis  = np.arange(N) / FS            # time axis in seconds
k          = np.arange(L)                  # sample index axis for one beat

print(f'ECG      shape={ecg_signal.shape}  dtype={ecg_signal.dtype}')
print(f'         min={ecg_signal.min():.4f}  max={ecg_signal.max():.4f}')
print(f'Template shape={template.shape}    dtype={template.dtype}')
print(f'         min={template.min():.4f}  max={template.max():.4f}')
print(f'Duration = {N/FS:.0f} s   |   Beats (ideal) = {N//L}')


---
## Part (a) — Reading the Signal  [0.75%]

In [ ]:
beat_period = 0.8   # seconds (given: one healthy P-QRS-T cycle)

# (i) Total clip duration
clip_duration = N / FS
print(f'(i)  Clip duration        = {N}/{FS} = {clip_duration:.0f} s')

# (ii) Heart rate and samples per beat
heart_rate_bpm   = 60.0 / beat_period
samples_per_beat = int(beat_period * FS)
print(f'(ii) Heart rate           = 60/0.8 = {heart_rate_bpm:.0f} BPM')
print(f'     Samples per beat     = 0.8 × {FS} = {samples_per_beat} samples')

# (iii) Fundamental frequency
f0 = 1.0 / beat_period
print(f'(iii)Fundamental freq f0  = 1/0.8 = {f0:.2f} Hz')


In [ ]:
# Plot: full ECG — real data
fig, ax = plt.subplots(figsize=(13, 3.5))
TRUE_ARR   = 2400        # arrhythmia starts at sample 2400 (t=9.6 s) — from data
TRUE_ARR_T = TRUE_ARR / FS

ax.plot(time_axis[:TRUE_ARR], ecg_signal[:TRUE_ARR],
        color=C_HEALTHY, lw=0.85, label=f'Healthy region (0–{TRUE_ARR_T:.1f} s)')
ax.plot(time_axis[TRUE_ARR:], ecg_signal[TRUE_ARR:],
        color=C_ARRHY,   lw=0.85, label=f'Arrhythmia region ({TRUE_ARR_T:.1f}–20 s)')
ax.axvspan(0, L/FS, alpha=0.20, color=C_TEMPLATE,
           label=f'One beat window  L={L} samples = 0.8 s')
ax.axvline(TRUE_ARR_T, color='white', lw=1, ls=':', alpha=0.7,
           label=f'Arrhythmia start ({TRUE_ARR_T:.1f} s)')
ax.set(title='Part (a): Full 20-second ECG Recording  [Real Dataset  fs=250 Hz, N=5000]',
       xlabel='Time (s)', ylabel='Amplitude', xlim=(0, 20))
ax.legend(loc='upper right', ncol=2); ax.grid(True)
plt.tight_layout(); plt.show()


---
## Part (b) — Frequency Domain  [0.75%]

In [ ]:
TRUE_ARR = 2400

# (i) Magnitude spectrum of healthy region only
healthy_seg = ecg_signal[:TRUE_ARR]
X_h   = np.fft.rfft(healthy_seg, n=TRUE_ARR)
freqs = np.fft.rfftfreq(TRUE_ARR, d=1/FS)

print('(i)  |X(f)| forms a discrete harmonic line spectrum with peaks at')
print(f'     k·f0 = k×{f0:.2f} Hz  (k=1,2,3,…) — comb-like profile.')
print()
print('(ii) The QRS complex is the source of high-frequency energy.')
print('     Its sharp, narrow peak ↔ broad spectral sidebands (Fourier duality).')
print('     P and T waves are broad/smooth → confined to low frequencies.')
print()
hr_new  = 150.0
f0_new  = hr_new / 60.0
print(f'(iii)At {hr_new} BPM: f0_new = {f0_new:.2f} Hz — harmonic spacing DOUBLES.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

# Left: |X(f)| real ECG healthy region
ax = axes[0]
ax.plot(freqs, np.abs(X_h), color=C_HEALTHY, lw=0.9)
ax.set_xlim(0, 30)
for ki in range(1, 12):
    hf = ki * 1.25
    if hf <= 30: ax.axvline(hf, color=C_ONSET, lw=0.7, ls='--', alpha=0.7)
ax.text(1.4, ax.get_ylim()[1]*0.85, 'Harmonics k·f₀', color=C_ONSET, fontsize=8)
ax.set(title='(b-i) |X(f)| Real ECG — Harmonic Line Spectrum',
       xlabel='Frequency (Hz)', ylabel='|X(f)|'); ax.grid(True)

# Right: 75 bpm vs 150 bpm harmonic spacing
ax = axes[1]
ax.plot(freqs, np.abs(X_h), color=C_HEALTHY, lw=0.9, label='75 bpm  f₀=1.25 Hz')
for ki in range(1, 8):
    ax.axvline(ki*1.25, color=C_HEALTHY, lw=0.7, ls=':', alpha=0.4)
    ax.axvline(ki*2.50, color=C_ARRHY,   lw=0.7, ls=':', alpha=0.7)
ax.axvline(-1, color=C_ARRHY, lw=1.5, ls=':', label='150 bpm  f₀=2.5 Hz (spacing doubles)')
ax.set_xlim(0, 20)
ax.set(title='(b-iii) Harmonic Spacing Doubles at 150 bpm',
       xlabel='Frequency (Hz)', ylabel='|X(f)|')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


---
## Part (c) — Windowing  [1.0%]

In [ ]:
optimal = 200; short = 80; long = 600

print(f'(i)  Optimal window = {optimal} samples = {optimal/FS:.2f} s = one complete P-QRS-T cycle.')
print(f'     Place it at the start of a healthy beat (e.g. sample 0).')
print()
print(f'(ii) {short}-sample window = {short/FS:.2f} s: captures QRS only — '
      f'T wave (and part of P wave) TRUNCATED.')
print(f'     {long}-sample window = {long/FS:.1f} s: spans ~{long/optimal:.0f} beats — '
      f'template matches multiple beats, blurring the score.')


In [ ]:
seg0 = ecg_signal[0:L]
w80  = np.zeros(L); w80[:80] = 1.0
w200 = np.ones(L)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

# Real template
ax = axes[0]
ax.plot(k, template, color=C_TEMPLATE, lw=2)
ax.fill_between(k, template, alpha=0.22, color=C_TEMPLATE)
peak_q = int(np.argmax(template))
ax.annotate('QRS', (peak_q, template[peak_q]+0.05), color=C_ONSET,
            fontsize=11, fontweight='bold', ha='center')
ax.annotate('P',  (20, 0.15), color=C_ONSET, fontsize=11, fontweight='bold')
ax.annotate('T',  (155, 0.18), color=C_ONSET, fontsize=11, fontweight='bold')
ax.set(title='(c) Real Template t[k]  (L=200)', xlabel='Sample k', ylabel='Amplitude')
ax.grid(True)

# 80-sample window
ax = axes[1]
ax.plot(k, seg0,     color=C_HEALTHY, lw=1.2, label='ECG segment')
ax.plot(k, w80*1.1,  color='orange',  lw=1.8, ls='--', label='80-sample window')
ax.fill_between(k, seg0*w80, alpha=0.35, color='orange', label='Windowed result')
ax.set(title='80-sample window — T wave MISSING', xlabel='Sample k')
ax.legend(fontsize=7.5); ax.grid(True)

# 200-sample window
ax = axes[2]
ax.plot(k, seg0,       color=C_HEALTHY, lw=1.2, label='ECG segment')
ax.plot(k, w200*1.1,   color='#b388ff', lw=1.8, ls='--', label='200-sample window')
ax.fill_between(k, seg0*w200, alpha=0.28, color='#b388ff', label='Full beat captured')
ax.set(title='200-sample window — Complete P-QRS-T', xlabel='Sample k')
ax.legend(fontsize=7.5); ax.grid(True)

plt.tight_layout(); plt.show()


---
## Part (d) — Template Correlation  [1.5%]

In [ ]:
t_norm_ = np.linalg.norm(template)

def rho_score(template, segment):
    xn = np.linalg.norm(segment)
    return float(np.dot(template, segment) / (t_norm_ * xn)) if xn > 1e-10 else 0.0

# (i) Range
print('(i)  ρ(m) ∈ [−1, +1].  Perfect positive match → ρ = +1.')
print()

# (ii) Normalization effect
seg_h     = ecg_signal[0:L]
seg_double = 2 * seg_h
r_h  = rho_score(template, seg_h)
r_d  = rho_score(template, seg_double)
un_h = float(np.dot(template, seg_h))
un_d = float(np.dot(template, seg_double))
print(f'(ii) Un-normalised  normal  = {un_h:.4f}')
print(f'     Un-normalised  doubled = {un_d:.4f}  (doubled!)')
print(f'     Normalised ρ   normal  = {r_h:.4f}')
print(f'     Normalised ρ   doubled = {r_d:.4f}  (identical — shape-only)')
print()

# (iii) Arrhythmia / inverted beat
TRUE_ARR = 2400
inv_seg  = ecg_signal[TRUE_ARR:TRUE_ARR+L]
r_inv    = rho_score(template, inv_seg)
print(f'(iii)ρ for first arrhythmia beat = {r_inv:.4f}  (strongly negative → inverted/abnormal)')


In [ ]:
TRUE_ARR   = 2400
TRUE_ARR_T = TRUE_ARR / FS

# Smooth ρ at every sample
rho_all = np.zeros(N - L)
for m in range(N - L):
    seg = ecg_signal[m : m + L]
    xn  = np.linalg.norm(seg)
    rho_all[m] = float(np.dot(template, seg) / (t_norm_ * xn)) if xn > 1e-10 else 0.0
t_rho = np.arange(len(rho_all)) / FS

fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))

ax = axes[0]
ax.plot(t_rho[:TRUE_ARR], rho_all[:TRUE_ARR], color=C_HEALTHY, lw=0.85, label='ρ(m) healthy')
ax.plot(t_rho[TRUE_ARR:], rho_all[TRUE_ARR:], color=C_ARRHY,   lw=0.85, label='ρ(m) arrhythmia')
ax.axhline(0.5, color=C_ONSET, lw=1.8, ls='--', label='Threshold = 0.5')
ax.axhline(1.0, color='white', lw=0.5, ls=':', alpha=0.4)
ax.fill_between(t_rho, rho_all, 0.5,
                where=(rho_all < 0.5) & (t_rho >= TRUE_ARR_T),
                color=C_ARRHY, alpha=0.22, label='Arrhythmia region')
ax.set(title='(d) Normalised Correlation ρ(m) — Real Data',
       xlabel='Time (s)', ylabel='ρ(m)', ylim=(-1.35, 1.45), xlim=(0, 20))
ax.legend(loc='lower left', ncol=2, fontsize=7.5); ax.grid(True)

ax = axes[1]
inv_seg = ecg_signal[TRUE_ARR:TRUE_ARR+L]
r_inv   = rho_score(template, inv_seg)
ax.plot(k, template, color=C_TEMPLATE, lw=2,   label='Template t[k]')
ax.plot(k, inv_seg,  color=C_ARRHY,   lw=1.8, label=f'Arrhythmia beat  ρ={r_inv:.2f}')
ax.fill_between(k, template, inv_seg, alpha=0.12, color='purple')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.set(title=f'(d-iii) Arrhythmia Beat vs Template  [ρ={r_inv:.2f}]',
       xlabel='Sample k', ylabel='Amplitude')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


---
## Part (e) — Onset Detection & Spectrogram  [1.5%]

In [ ]:
TRUE_ARR   = 2400
TRUE_ARR_T = TRUE_ARR / FS

print('(i)  Onset rule: declare arrhythmia at first m where ρ(m) < threshold.')
print('     Threshold too HIGH → false alarms in healthy region.')
print('     Threshold too LOW  → misses early/mild arrhythmic beats.')
print()
print('(ii) Healthy spectrogram: steady horizontal harmonic bands at 1.25, 2.5, 3.75 Hz …')
print('     Arrhythmia region : bands shift / dissolve into broadband noise.')
print()
print('(iii)Correlation fires on the EXACT first bad beat (sample precision).')
print('     Spectrogram needs several beats to show frequency change (time-smearing).')
print('     → For pinpointing exact onset: trust the CORRELATION plot.')


In [ ]:
# Already computed rho_all above; re-use it
fig, ax = plt.subplots(figsize=(13, 4.0))
ax.plot(t_rho[:TRUE_ARR], rho_all[:TRUE_ARR], color=C_HEALTHY, lw=0.85, label='ρ(m) healthy')
ax.plot(t_rho[TRUE_ARR:], rho_all[TRUE_ARR:], color=C_ARRHY,   lw=0.85, label='ρ(m) arrhythmia')
ax.axhline(0.5, color=C_ONSET, lw=1.8, ls='--', label='Threshold = 0.5')
ax.fill_between(t_rho, rho_all, 0.5,
                where=(rho_all < 0.5) & (t_rho >= TRUE_ARR_T),
                color=C_ARRHY, alpha=0.22, label='Below-threshold region')
ax.axvline(TRUE_ARR_T, color='white', lw=1, ls=':', alpha=0.6,
           label=f'Arrhythmia start ({TRUE_ARR_T:.1f} s)')
ax.set(title='(e) ρ(m) for Onset Detection — Real Data',
       xlabel='Time (s)', ylabel='ρ(m)', ylim=(-1.35, 1.45), xlim=(0, 20))
ax.legend(loc='lower left', ncol=3); ax.grid(True)
plt.tight_layout(); plt.show()


---
## Part (f) — Sampling & Aliasing  [0.5%]

In [ ]:
f_max = 40
fs_min = 2 * f_max
fs_low = 50
fold   = fs_low / 2

print(f'(i)  Nyquist fs_min = 2 × {f_max} = {fs_min} Hz')
print()
print(f'(ii) At fs\'={fs_low} Hz  (< {fs_min} Hz): Nyquist condition VIOLATED.')
print(f'     Frequencies above {fold} Hz ALIAS into the lower spectrum.')
print(f'     QRS content up to {f_max} Hz folds back → distorts QRS shape → detector fails.')
print()
print(f'(iii)Fix: keep fs ≥ {fs_min} Hz (original 250 Hz is safe).')
print('     Cost: larger storage, higher power draw on the wearable, more compute.')


In [ ]:
fs_orig = 250
t_orig  = np.linspace(0, 0.8, fs_orig, endpoint=False)
qrs_sharp = np.exp(-0.5 * ((t_orig - 0.4) / 0.008) ** 2)
t_low     = t_orig[::5]
qrs_aliased = qrs_sharp[::5]

# Anti-aliased version
cutoff = 25; nyq = 0.5 * fs_orig
b, a   = butter(4, cutoff / nyq, btype='low')
qrs_filtered = filtfilt(b, a, qrs_sharp)
qrs_safe = qrs_filtered[::5]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

ax = axes[0]
ax.plot(t_orig*1000, qrs_sharp,    color=C_HEALTHY, lw=2, label='Original QRS at 250 Hz')
ax.stem(t_low*1000,  qrs_aliased,  linefmt='r--', markerfmt='ro',
        basefmt=' ', label='Aliased at 50 Hz (no filter)')
ax.set(title='(f-ii) Aliasing Distorts the QRS Spike at 50 Hz',
       xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(); ax.grid(True)

ax = axes[1]
f_demo  = np.linspace(0, 60, 500)
psd     = np.exp(-(f_demo / 15) ** 2)
ax.fill_between(f_demo[f_demo <= 40], psd[f_demo <= 40],
                alpha=0.5, color=C_HEALTHY, label='QRS energy (0–40 Hz)')
ax.axvline(25, color=C_ONSET, lw=2, ls='--', label='Nyquist @ 50 Hz = 25 Hz')
ax.fill_between(f_demo[(f_demo > 25) & (f_demo <= 40)],
                psd[(f_demo > 25) & (f_demo <= 40)],
                alpha=0.65, color=C_ARRHY, label='Energy that ALIASES (25–40 Hz)')
ax.set(title='(f-i/ii) Nyquist Limit & Aliasing Region',
       xlabel='Frequency (Hz)', ylabel='Power')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


---
## Part (g) — `find_onset()` Implementation  [1.5%]

$$\rho(m) = \frac{\sum_k t[k]\,x[m+k]}{\|t\|\;\|x_m\|}$$

Beat-by-beat stride = L = 200 samples. Return first m where ρ < threshold, or -1.

In [ ]:
def find_onset(ecg_signal, template, threshold=0.5):
    """
    Detect arrhythmia onset via beat-by-beat normalised correlation.

    Parameters
    ----------
    ecg_signal : ndarray, shape (N,)
        Full ECG recording from patient_ecg.npy.
    template   : ndarray, shape (L,)
        Clean healthy template from template.npy.
    threshold  : float
        Correlation strictly below which arrhythmia is declared.
        Default = 0.5.

    Returns
    -------
    m         : int
        Sample index of first beat where ρ(m) < threshold. −1 if never.
    positions : ndarray of int
        All beat start indices evaluated.
    rho_vals  : ndarray of float
        Normalised correlation at each beat.
    """
    N_sig  = len(ecg_signal)
    L_     = len(template)
    t_norm = np.linalg.norm(template)   # ||t|| — computed once

    positions = []
    rho_vals  = []
    m = 0

    while m + L_ <= N_sig:
        segment = ecg_signal[m : m + L_]    # window of length L
        x_norm  = np.linalg.norm(segment)   # ||x_m||

        if x_norm < 1e-10:                  # guard: flat/silent segment
            rho = 0.0
        else:
            rho = float(np.dot(template, segment) / (t_norm * x_norm))

        positions.append(m)
        rho_vals.append(rho)

        if rho < threshold:                 # ← first breach → onset found
            return m, np.array(positions), np.array(rho_vals)

        m += L_                             # jump one full beat forward

    return -1, np.array(positions), np.array(rho_vals)


In [ ]:
# ── Run on real data ──────────────────────────────────────────────────────────
THRESHOLD = 0.5
onset_m, beat_pos, beat_rho = find_onset(ecg_signal, template, threshold=THRESHOLD)

if onset_m != -1:
    onset_time = onset_m / FS
    print(f'Arrhythmia onset detected:')
    print(f'  → Sample index m   = {onset_m}')
    print(f'  → Time             = {onset_time:.3f} s')
    print(f'  → ρ at onset beat  = {beat_rho[-1]:.4f}  (below threshold {THRESHOLD})')
else:
    print('Threshold never breached — no arrhythmia detected.')

print(f'\nTotal beats evaluated  : {len(beat_pos)}')
print(f'Beats above threshold  : {(beat_rho >= THRESHOLD).sum()}  (healthy)')
print(f'Beats below threshold  : {(beat_rho <  THRESHOLD).sum()}  (arrhythmic)')


In [ ]:
TRUE_ARR   = 2400
TRUE_ARR_T = TRUE_ARR / FS

# ── Panel figure: ECG + beat-by-beat bar + beat comparison ───────────────────
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.30,
                        left=0.07, right=0.97, top=0.92, bottom=0.07)

# Panel 1: full ECG with onset marker
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(time_axis[:TRUE_ARR], ecg_signal[:TRUE_ARR],
         color=C_HEALTHY, lw=0.85, label='Healthy region')
ax1.plot(time_axis[TRUE_ARR:], ecg_signal[TRUE_ARR:],
         color=C_ARRHY,   lw=0.85, label='Arrhythmia region')
ax1.axvline(onset_time, color=C_ONSET, lw=2, ls='--',
            label=f'Onset detected  m={onset_m}  t={onset_time:.2f} s')
ax1.axvline(TRUE_ARR_T, color='white', lw=1, ls=':', alpha=0.6,
            label=f'True arrhythmia start ({TRUE_ARR_T:.1f} s)')
ax1.set(title='Part (g): ECG with Detected Onset  [Real Dataset]',
        xlabel='Time (s)', ylabel='Amplitude', xlim=(0, 20))
ax1.legend(loc='upper right', ncol=2); ax1.grid(True)

# Panel 2: beat-by-beat bar chart
ax2 = fig.add_subplot(gs[1, 0])
beat_times = beat_pos / FS
bar_cols   = [C_HEALTHY if r >= THRESHOLD else C_ARRHY for r in beat_rho]
ax2.bar(beat_times, beat_rho, width=0.55, color=bar_cols, alpha=0.88)
ax2.axhline(THRESHOLD, color=C_ONSET, lw=1.8, ls='--', label=f'Threshold={THRESHOLD}')
ax2.axvline(onset_time, color=C_ONSET, lw=2.2,
            label=f'Onset m={onset_m} ({onset_time:.2f} s)')
ax2.set(title='Beat-by-Beat ρ  (stride = L = 200 samples)',
        xlabel='Time (s)', ylabel='ρ(m)', ylim=(-1.4, 1.4))
ax2.text(0.03, 0.06, f'Onset: sample {onset_m}  ({onset_time:.2f} s)',
         transform=ax2.transAxes, color=C_ONSET, fontsize=9, fontweight='bold')
ax2.legend(); ax2.grid(True)

# Panel 3: template vs healthy vs onset beat
ax3 = fig.add_subplot(gs[1, 1])
seg_h = ecg_signal[0 : L]
seg_o = ecg_signal[onset_m : onset_m + L]
rh = float(np.dot(template, seg_h) / (np.linalg.norm(template) * np.linalg.norm(seg_h)))
ro = float(np.dot(template, seg_o) / (np.linalg.norm(template) * np.linalg.norm(seg_o)))
ax3.plot(k, template, color=C_TEMPLATE, lw=2.0, label='Template t[k]')
ax3.plot(k, seg_h,    color=C_HEALTHY,  lw=1.5, ls='--', label=f'Healthy beat  ρ={rh:.2f}')
ax3.plot(k, seg_o,    color=C_ARRHY,    lw=1.5,           label=f'Onset beat    ρ={ro:.2f}')
ax3.axhline(0, color='white', lw=0.5, alpha=0.35)
ax3.set(title='Template vs Healthy vs Onset Beat  (Real Data)',
        xlabel='Sample k', ylabel='Amplitude')
ax3.legend(); ax3.grid(True)

fig.suptitle('Part (g): find_onset() — Beat-by-Beat Detector  [Real Dataset]',
             fontsize=12, fontweight='bold', y=0.97)
plt.show()


---
## Part (h) — Spectrogram  [0.5%]

**Window choice:** `nperseg = 200` samples (0.8 s = one beat period)

> **One-sentence justification:** `nperseg = 200` was chosen because it equals exactly one healthy beat period (0.8 s), giving a DFT frequency resolution of 250/200 = **1.25 Hz = f₀**, so each cardiac harmonic occupies its own DFT bin, producing the steady horizontal bands visible in the healthy region.

In [ ]:
NPERSEG  = 200          # one beat = 0.8 s → Δf = 1.25 Hz = f0
NOVERLAP = NPERSEG // 2 # 50 % overlap

freqs_s, times_s, Sxx = sp_spectrogram(
    ecg_signal, fs=FS,
    nperseg=NPERSEG, noverlap=NOVERLAP,
    scaling='spectrum'
)
print(f'nperseg  = {NPERSEG} samples  ({NPERSEG/FS:.2f} s)')
print(f'Freq resolution Δf = {FS}/{NPERSEG} = {FS/NPERSEG:.2f} Hz  = f0')
print(f'Spectrogram shape: freqs={freqs_s.shape}, times={times_s.shape}, Sxx={Sxx.shape}')


In [ ]:
TRUE_ARR   = 2400
TRUE_ARR_T = TRUE_ARR / FS

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5), gridspec_kw={'hspace': 0.40})

# Top: ECG reference
ax = axes[0]
ax.plot(time_axis[:TRUE_ARR], ecg_signal[:TRUE_ARR],
        color=C_HEALTHY, lw=0.85, label='Healthy')
ax.plot(time_axis[TRUE_ARR:], ecg_signal[TRUE_ARR:],
        color=C_ARRHY,   lw=0.85, label='Arrhythmia')
ax.axvline(onset_time, color=C_ONSET, lw=2, ls='--',
           label=f'Onset @ {onset_time:.2f} s')
ax.axvline(TRUE_ARR_T, color='white', lw=1, ls=':', alpha=0.5)
ax.set(title='(h) ECG Reference  [Real Dataset]',
       xlabel='Time (s)', ylabel='Amplitude', xlim=(0, 20))
ax.legend(loc='upper right', ncol=3); ax.grid(True)

# Bottom: spectrogram
ax = axes[1]
im = ax.pcolormesh(times_s, freqs_s,
                   10 * np.log10(Sxx + 1e-12),
                   shading='gouraud', cmap='inferno', vmin=-55, vmax=0)
ax.set_ylim(0, 40)
ax.axvline(onset_time, color=C_ONSET, lw=2, ls='--',
           label=f'Onset @ {onset_time:.2f} s')
ax.axvline(TRUE_ARR_T, color='white', lw=1, ls=':', alpha=0.55,
           label=f'True start ({TRUE_ARR_T:.1f} s)')
for i, hf in enumerate([1.25, 2.50, 3.75, 5.00, 6.25]):
    ax.axhline(hf, color='cyan', lw=0.8, ls='--', alpha=0.6)
    ax.text(0.3, hf+0.25, f'f₀={hf}Hz' if i == 0 else f'{hf}Hz',
            color='cyan', fontsize=7.5, alpha=0.85)
cbar = fig.colorbar(im, ax=ax, pad=0.01)
cbar.set_label('Power (dB)', color=C_TEXT, fontsize=9)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=C_TEXT, fontsize=8)
ax.set(title=(f'(h) Spectrogram  [nperseg={NPERSEG}, Δf={FS/NPERSEG:.2f} Hz = f₀]  '
              f'Horizontal bands=healthy; Disrupted=arrhythmia'),
       xlabel='Time (s)', ylabel='Frequency (Hz)', xlim=(0, 20))
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.suptitle('Part (h): STFT Spectrogram  [nperseg=200 = one beat period, Real Dataset]',
             fontsize=12, fontweight='bold', y=0.99)
plt.show()


---
## Summary of Results  [Real Dataset]

| Part | Key Result |
|------|------------|
| (a) | Duration=**20 s**, HR=**75 BPM**, L=**200 samples**, f₀=**1.25 Hz** |
| (b) | Harmonic line spectrum; QRS → high-freq; at 150 BPM f₀ doubles to 2.5 Hz |
| (c) | Optimal window = **200 samples**; 80 cuts T-wave; 600 captures 3 beats |
| (d) | ρ∈[−1,+1]; normalization removes amplitude bias; arrhythmia beat ρ≈−0.99 |
| (e) | Onset at ρ<0.5; spectrogram shows band breakdown; correlation is more precise |
| (f) | fs_min=**80 Hz**; at 50 Hz QRS aliases above 25 Hz → detector fails |
| (g) | **`find_onset()` → m=2400, t=9.60 s, ρ=−0.9879** |
| (h) | **nperseg=200**: Δf=1.25 Hz=f₀ → steady harmonic bands in healthy region |
